In [ ]:
%matplotlib inline

# Sugeno lambda FM

I built the following on top of the Sugeno integral, you can do the same for the Choquet integral

In [ ]:
# lets import some libs
import numpy as np
import itertools
import matplotlib
import matplotlib.pyplot as plt
from itertools import combinations

# lets make our sugeno integral class
class SugenoIntegral:

    def __init__(self):
        """
            init function
        """
        
        self.N = 0    
        self.fm = []  
        self.g = []   
        
    def evaluate(self, x):
        """
            evaluate the Sugeno integral
        """

        hs = np.zeros(x.size)
        gs = np.zeros(x.size)
        
        print("Input:",x)
                
        # do our sort
        pi_i = np.argsort(x)[::-1] + 1
        
        print("Sort:",pi_i)
        
        # do the first calculation
        h = x[pi_i[0] - 1]
        g = self.fm[ str(pi_i[:1]) ]
        o = min( h , g )
        print("[h,g] Step 1 :",h,g)
        print(str(pi_i[:1]))
        hs[0] = h
        gs[0] = g
        
        # do the other N-1 terms
        for i in range(1, self.N):
            h = x[pi_i[i] - 1]            
            g = self.fm[str(np.sort(pi_i[:i + 1]))]
            hs[i] = h
            gs[i] = g
            print("[h,g] Step",i+1,":",h,g)
            print(str(np.sort(pi_i[:i + 1])))
            # our calculation, namely, max of the mins
            o = max( o, min( h, g ) )
            
        return o, hs, gs

    def calc_sugeno_lambda_fm(self,densities):
        """
            calc the sugeno lambda FM
            :densities: the input densities
            :return: return's lambda
        """        
    
        n = len(densities)
        print(densities)
        
        coeff = np.zeros(n)
        
        # calc the product terms
        ind = 0
        for i in range(n,1,-1):
            c = combinations(range(1,n+1),i)
            r = 0
            for i in list(c):
                inds = np.asarray(i) - 1
                r = r + np.prod(densities[inds])
            coeff[ind] = r
            ind = ind + 1
        # last term
        coeff[n-1] = np.sum(densities)-1

        print(coeff)
        
        l_possible = np.roots(coeff)
        
        print(l_possible)
        
        # I need to find how to get the real's in Python?
        lamb = np.max( np.real( l_possible ) )
        
        return lamb
    
    def get_keys_index(self):
        """
            sets up a dictionary for referencing the FM
            :return: keys to the dictionary
        """

        vls = np.arange(1, self.N + 1)
        count = 0
        Lattice = {}
        for i in range(0, self.N):
            Lattice[str(np.array([vls[i]]))] = count
            count = count + 1
        for i in range(2, self.N + 1):
            A = np.array(list(itertools.combinations(vls, i)))
            for latt_pt in A:
                Lattice[str(latt_pt)] = count
                count = count + 1
        return Lattice    
    
    def produce_lattice(self):
        """
            makes a nice little ole data structure for us to index our fuzzy measure 
        """

        index_keys = self.get_keys_index()
        Lattice = {}
        for key in index_keys.keys():
            Lattice[key] = self.g[index_keys[key]]
        return Lattice

Lets calc that lambda value

In [ ]:
# create our integral
sugeno = SugenoIntegral()

# make some densities
dens = np.asarray( [0.6, 0.7, 0.3] )

# calc lambda
lamb = sugeno.calc_sugeno_lambda_fm( dens )
print(lamb)

If you want the full FM, then you need to walk through and do all your individual calcs

$g_{\lambda}(A \cup B) = g_{\lambda}(A) + g_{\lambda}(B) + \lambda g_{\lambda}(A) g_{\lambda}(B)$

e.g., calc $\{x_1, x_2\}$ via $A=\{x_1\}$ and $B=\{x_2\}$. Then, $\{x_1, x_2, x_3\}$ via $A=\{x_1,x_2\}$ and $B=\{x_3\}$ and etc.